## Imports

In [ ]:
import os
import re
import pandas as pd
import numpy as np

In [ ]:
import ollama
import wikipedia
from ddgs import DDGS
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

In [ ]:
#! ollama pull mistral

## Prepare questions

In [3]:
target_langs = [
    "en-GB",  # English (United Kingdom)
    "en-US",  # English (United States)
    "en-AU",  # English (Australia)

    "es-ES",  # Spanish (Spain)
    "es-MX",  # Spanish (Mexico)
    "es-EC",  # Spanish (Ecuador)

    "zh-CN",  # Chinese (China)
    "zh-SG",  # Singaporean Mandarin (Singapore)

    "ar-DZ",  # Arabic (Algeria)
    "ar-EG",  # Arabic (Egypt)
    "ar-MA",  # Arabic (Morocco)
    "ar-SA",  # Arabic (Saudi Arabia)

    "am-ET",  # Amharic (Ethiopia)
    "ha-NG",  # Hausa (Northern Nigeria)
    "as-AS",  # Assamese (Assam, India)
    "az-AZ",  # Azerbaijani (Azerbaijan)
    "id-ID",  # Indonesian (Indonesia)
    "su-JB",  # Sundanese (West Java, Indonesia)
    "fa-IR",  # Persian/Farsi (Iran)
    "ko-KP",  # Korean (North Korea)
    "ko-KR",  # Korean (South Korea)
    "el-GR",  # Greek (Greece)
    "ja-JP",  # Japanese (Japan)
    "tl-PH",  # Tagalog (Philippines)
    "ta-LK",  # Tamil (Sri Lanka)
    "ta-SG",  # Tamil (Singapore)
    "ms-SG",  # Malay (Singapore)
    "bg-BG",  # Bulgarian (Bulgaria)
    "fr-FR",  # French (France)
    "ga-IE",  # Irish (Ireland)
    "eu-ES",  # Basque (Spain)
]

id_country = {
    "GB": "United Kingdom",
    "US": "United States",
    "AU": "Australia",
    "ES": "Spain",
    "MX": "Mexico",
    "EC": "Ecuador",
    "SG": "Singapore",
    "CN": "China",
    "DZ": "Algeria",
    "ET": "Ethiopia",
    "NG": "Nigeria",
    "AS": "Assam, India",
    "AZ": "Azerbaijan",
    "ID": "Indonesia",
    "JB": "West Java, Indonesia",   # Sundanese region
    "IR": "Iran",
    "KP": "North Korea",
    "KR": "South Korea",
    "GR": "Greece",
    "JP": "Japan",
    "PH": "Philippines",
    "LK": "Sri Lanka",
    "BG": "Bulgaria",
    "FR": "France",
    "IE": "Ireland",
}


In [ ]:
df_track_1 = pd.read_csv('track_1_saq_input 2.tsv', sep='\t')
df_track_1.head()

In [ ]:
df_track_2 = pd.read_csv('track_2_mcq_input 2.tsv', sep='\t')
df_track_2.head()

In [ ]:
mask = df_track_1['id'].str.startswith(tuple(target_langs))

df_filtered_track_1 = df_track_1[mask]
df_filtered_track_1.head()

In [ ]:
mask = df_track_2['id'].str.startswith(tuple(target_langs))

df_filtered_track_2 = df_track_2[mask]
df_filtered_track_2.head()

# Track 1

## Calculate threshold for countries

In [ ]:
def list_db_countries(db_root="."):
    return sorted([
        d.replace("db_", "")
        for d in os.listdir(db_root)
        if d.startswith("db_") and os.path.isdir(os.path.join(db_root, d))
    ])

def country_from_id(qid: str):
    # "am-ET_001" -> "ET"
    lang_code = qid.split("_")[0]  # "am-ET"
    if "-" in lang_code:
        return lang_code.split("-")[-1]
    return lang_code


In [ ]:
def compute_distance_stats_all_countries(
    df,
    db_root=".",
    n_per_country=200,
    seed=42,
    k=1,
):
    rng = np.random.default_rng(seed)

    countries = list_db_countries(db_root)
    rows = []

    df = df.copy()
    df["country"] = df["id"].apply(country_from_id)

    for cc in countries:
        sub = df[df["country"] == cc]
        if len(sub) == 0:
            continue

        take = min(n_per_country, len(sub))
        sub_sample = sub.sample(n=take, random_state=seed)

        persist_dir = os.path.join(db_root, f"db_{cc}")
        try:
            db = Chroma(
                persist_directory=persist_dir,
                embedding_function=OllamaEmbeddings(model="mistral"),
            )
        except Exception as e:
            rows.append({
                "country": cc,
                "n_questions": take,
                "error": f"failed to load db: {e}"
            })
            continue

        dists = []
        for q in sub_sample["text"].tolist():
            try:
                ds = db.similarity_search_with_score(q, k=k)
                if ds:
                    dists.append(ds[0][1])  # best distance
            except Exception:
                continue

        if len(dists) == 0:
            rows.append({
                "country": cc,
                "n_questions": take,
                "n_scored": 0,
                "error": "no distances collected"
            })
            continue

        dists = np.array(dists, dtype=float)
        rows.append({
            "country": cc,
            "n_questions": take,
            "n_scored": len(dists),
            "min": float(dists.min()),
            "p10": float(np.percentile(dists, 10)),
            "p25": float(np.percentile(dists, 25)),
            "p50": float(np.percentile(dists, 50)),
            "p75": float(np.percentile(dists, 75)),
            "p90": float(np.percentile(dists, 90)),
            "p95": float(np.percentile(dists, 95)),
            "max": float(dists.max()),
            "error": ""
        })

    stats_df = pd.DataFrame(rows).sort_values(["error", "country"]).reset_index(drop=True)

    all_dists = []
    for _, r in stats_df.iterrows():
        pass

    return stats_df


In [ ]:
stats_df = compute_distance_stats_all_countries(
    df_filtered_track_1,
    db_root=".",
    n_per_country=200,
    seed=42
)

stats_df.head(20)

## Experiments with models

In [ ]:
def web_search(query: str, region: str | None = None, max_results: int = 5) -> str:
    """
    Web search using the new `ddgs` package (DuckDuckGo Search).
    Returns concatenated snippets from search results.
    """
    try:
        snippets = []
        with DDGS() as ddgs:
            for r in ddgs.text(
                keywords=query,
                region=region,
                max_results=max_results,
            ):
                body = r.get("body", "")
                if body:
                    snippets.append(body)

        return "\n\n".join(snippets)

    except Exception:
        return ""



In [11]:
def load_or_create_db(country_code, texts):
    persist_dir = f"db_{country_code}"
    if os.path.exists(persist_dir):
        db = Chroma(persist_directory=persist_dir, embedding_function=OllamaEmbeddings(model="mistral"))
    else:
        splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
        docs = splitter.create_documents(texts)
        db = Chroma.from_documents(docs, embedding=OllamaEmbeddings(model="mistral"), persist_directory=persist_dir)
        db.persist()
    return db

In [37]:
threshold_by_country = dict(zip(stats_df["country"], stats_df["p10"]))  # или p25

def local_retrieve_with_gate(db, question, country_code, k=4):
    max_distance = threshold_by_country.get(country_code, None)
    if max_distance is None:
        return "", None, False

    docs_scores = db.similarity_search_with_score(question, k=k)
    if not docs_scores:
        return "", None, False

    best = min(score for _, score in docs_scores)

    if best > max_distance:
        return "", best, False

    context = "\n\n".join(doc.page_content for doc, _ in docs_scores)
    return context, best, True

In [ ]:
NO_ANSWER = "<NO_ANSWER>"

def _set_wiki_lang(lang_code: str | None):
    """Set Wikipedia language based on lang_code like 'en-US' -> 'en'."""
    if not lang_code:
        wikipedia.set_lang("en")
        return
    lang = lang_code.split("-")[0].lower()
    try:
        wikipedia.set_lang(lang)
    except Exception:
        wikipedia.set_lang("en")

def wiki_lookup(query: str, lang_code: str | None = None, sentences: int = 3) -> str:
    """Best-effort Wikipedia lookup: search -> summary of top hit."""
    _set_wiki_lang(lang_code)
    try:
        results = wikipedia.search(query, results=5)
        if not results:
            return ""
        title = results[0]
        return wikipedia.summary(title, sentences=sentences, auto_suggest=True)
    except Exception:
        return ""

def looks_encyclopedic(question: str) -> bool:
    """Heuristic: Wikipedia-first only for classic factual/encyclopedic questions."""
    q = (question or "").strip().lower()

    # strong signals for encyclopedic intent
    patterns = [
        r"\bcapital\b",
        r"\bpopulation\b",
        r"\blocated\b",
        r"\bfounded\b",
        r"\bborn\b",
        r"\bdied\b",
        r"\bpresident\b",
        r"\bprime minister\b",
        r"\bwhat is\b",
        r"\bwho is\b",
        r"\bwhen (did|was)\b",
        r"\bwhere is\b",
        r"\bofficial language\b",
    ]
    if any(re.search(p, q) for p in patterns):
        return True

    # dataset is often culture/food/habits → web-first
    cultureish = [
        "snack", "food", "eat", "drink", "popular", "usually", "typical",
        "street food", "beer", "children", "gift", "wedding", "holiday",
        "tradition", "custom", "wear", "music", "dance",
    ]
    if any(w in q for w in cultureish):
        return False

    # default: web-first
    return False

NO_ANSWER = "<NO_ANSWER>"

def normalize_no_answer(text: str) -> str:
    if text is None:
        return NO_ANSWER
    t = str(text).strip()

    low = t.lower()
    if "<no_answer>" in low or "no_answer" in low or "no answer" in low:
        return NO_ANSWER

    triggers = [
        "i don't know", "i do not know", "not sure", "uncertain", "unknown",
        "no information", "no data", "cannot answer", "unable to answer",
        "don't have enough information", "do not have enough information",
        "insufficient information", "not available", "no details",
    ]
    if any(x in low for x in triggers):
        return NO_ANSWER

    return t


In [53]:
def build_en_question_lookup(df):
    en = df[df["id"].str.startswith("en-")].copy()
    # key: "ET_001"
    en["key"] = en["id"].str.split("_").str[0].str.split("-").str[-1] + "_" + en["id"].str.split("_").str[1]
    return dict(zip(en["key"], en["text"]))

In [ ]:
#stats_df.to_csv("distance_stats_all_countries.tsv", sep="\t", index=False)
threshold_by_country = dict(zip(stats_df["country"], stats_df["p10"]))


In [54]:
def build_en_question_lookup(df):
    en = df[df["id"].str.startswith("en-")].copy()
    # key: "ET_001" из "en-ET_001"
    en["key"] = en["id"].str.split("_").str[0].str.split("-").str[-1] + "_" + en["id"].str.split("_").str[1]
    return dict(zip(en["key"], en["text"]))

en_q_lookup = build_en_question_lookup(df_filtered_track_1)

In [ ]:
qid_suffix = None

def ask_ollama_dynamic(
    question,
    qid=None,
    lang_code=None,
    system_prompt_base=("""You are a factual multilingual assistant for a question-answering benchmark.
Your goal is to give one short, correct answer for each question in its original language.
Instructions:
- Read the question carefully.
- Respond **only with the concise answer** — a word, number, name, or short phrase.
- Do not include explanations, reasoning, or extra words.
- Keep the answer in the **same language** as the question (Chinese → Chinese, English → English, Spanish → Spanish).
- If you are not confident you know the answer from your own knowledge, reply exactly: <NO_ANSWER>
Example outputs:
Q (English): What is the capital of France?
A: Paris
Q (Chinese): 中国的国庆节是哪一天？
A: 10月1日
Q (Spanish): ¿Cuál es el idioma oficial de España?
A: Español
Now answer the following question:
"""
        ),
    use_web: bool = True,
    use_local_db: bool = True,  # kept for backward compatibility; treated as "use Wikipedia"
    use_wiki = True,
):
    """
    Behavior:
    1) Ask the model with no external context. If it answers confidently -> return.
    2) If it returns <NO_ANSWER>, retrieve context:
       - Web-first for culture/food/habits questions (default for your SAQ dataset).
       - Wikipedia-first only for classic encyclopedic questions (heuristic).
    3) Ask the model again with retrieved context and return the final short answer.
    """

    country_name = None
    model_name = "mistral"

    # parse country/region code from lang_code
    country_code = None
    if lang_code and "-" in lang_code:
        country_code = lang_code.split("-")[-1]
        country_name = id_country.get(country_code)
    else:
        country_code = lang_code

    # choose model
    if lang_code and any(x in lang_code.lower() for x in ["zh", "cn", "tw", "ta"]):
        model_name = "deepseek-llm:67b"
    else:
        model_name = "mistral"

    # system prompt personalization
    if country_name:
        system_prompt = f"{system_prompt_base} You are from {country_name}."
    elif lang_code:
        system_prompt = f"{system_prompt_base} You are from the region represented by '{lang_code}'."
    else:
        system_prompt = system_prompt_base

    query_for_retrieval = question
    query_for_web = question

    if qid:
        # key: "ET_001"
        suffix = qid.split("_")[1]  # "001"
        key = f"{country_code}_{suffix}"
        en_q = en_q_lookup.get(key)
        if en_q:
            query_for_retrieval = en_q
            query_for_web = en_q

    # --- Step 1: model-only attempt (no RAG) ---
    messages_1 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query_for_retrieval},
    ]

    try:
        response_1 = ollama.chat(model=model_name, messages=messages_1)
        ans_1 = normalize_no_answer(response_1["message"]["content"])
    except Exception as e:
        return f"[Error: {e}]"

    if ans_1 != NO_ANSWER:
        return ans_1
    
    local_context = ""
    if use_local_db and country_code:
        try:
            db = Chroma(
                persist_directory=f"./db_{country_code}",
                embedding_function=OllamaEmbeddings(model="mistral")
            )

            local_context, best_dist, passed = local_retrieve_with_gate(db, query_for_retrieval, country_code)

            if passed and local_context.strip():
                user_content_local = (
                    "Answer ONLY if the answer is explicitly stated in the provided context. "
                    "If the context does not contain the answer, reply exactly: <NO_ANSWER>. "
                    "Do not guess.\n\n"
                    f"LOCAL KB CONTEXT:\n{local_context}\n\nQuestion: {query_for_retrieval}"
                )

                messages_local = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content_local},
                ]

                response_local = ollama.chat(model=model_name, messages=messages_local)
                ans_local = normalize_no_answer(response_local["message"]["content"])

                if ans_local != NO_ANSWER:
                    return ans_local
        except Exception:
            pass

    # --- Step 2: Retrieval (web vs wikipedia) ---
    web_context = ""
    wiki_context = ""

    wiki_first = looks_encyclopedic(question)

    # Treat use_local_db as "use Wikipedia" (kept name to avoid refactors)
    use_wiki = bool(use_local_db)

    if wiki_first and use_wiki:
        wiki_context = wiki_lookup(query_for_web, lang_code=lang_code, sentences=3)

    if use_web:
        region = country_code
        web_context = web_search(query_for_web, region=region, max_results=5)

    if (not wiki_first) and use_wiki and not wiki_context:
        # web-first path: try Wikipedia only if web didn't help
        if not web_context.strip():
            wiki_context = wiki_lookup(query_for_web, lang_code=lang_code, sentences=3)

    context_parts = []
    if web_context:
        context_parts.append(f"WEB SEARCH SNIPPETS:\n{web_context}")
    if wiki_context:
        context_parts.append(f"WIKIPEDIA EXCERPT:\n{wiki_context}")

    full_context_block = "\n\n".join(context_parts).strip()

    if not full_context_block:
        # nothing to ground on
        return NO_ANSWER

    user_content_2 = (
        "Answer ONLY if the answer is explicitly stated in the provided context. "
    "If the context does not contain the answer, reply exactly: <NO_ANSWER>. "
    "Do not guess.\n\n"
    f"{full_context_block}\n\nQuestion: {query_for_retrieval}"
)

    messages_2 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content_2},
    ]

    try:
        response_2 = ollama.chat(model=model_name, messages=messages_2, options={"temperature": 0, "num_predict": 32})
        ans_2 = normalize_no_answer(response_2["message"]["content"])
    except Exception as e:
        return f"[Error: {e}]"

    return ans_2


## Apply model

In [ ]:
results = []

for _, row in df_filtered_track_1.iterrows():
    qid = row["id"]
    question = row["text"]
    lang_code = qid.split("_")[0]

    answer = ask_ollama_dynamic(row["text"], qid=row["id"], lang_code=lang_code)

    results.append({"id": qid, "prediction": answer})

In [57]:
df_results = pd.DataFrame(results)

df_results.to_csv("track_1_saq_prediction_shiran_prompt_v1_with_local_db_evaluation_test.tsv", sep="\t", index=False)

In [51]:
df_results

,id,prediction
0,am-ET_001,Birr
1,am-ET_002,Birr
2,am-ET_003,Birr
3,am-ET_004,ETB (Ethiopian Birr)
4,am-ET_005,"Agriculture, Textiles, Coffee, and Manufacturing"
...,...,...
80,am-ET_081,Birr
81,am-ET_082,"""295.4 billion USD (as of 2021)"""
82,am-ET_083,"""Ethiopian restaurants with vegan options"""
83,am-ET_084,Lalibela Churches


# Track 2

In [ ]:
def ask_mcq_with_rag(question, options, lang_code=None):

    system_prompt = (
        "You answer multiple-choice questions. "
        "Always respond with exactly one capital letter: A, B, C, or D, "
        "corresponding to the best answer option. "
        "Do not explain your choice."
    )

    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])
    user_question = f"Question: {question}\n\nOptions:\n{options_text}\n\nAnswer with only one letter (A, B, C, or D)."
    raw_answer = ask_ollama_dynamic(
        question=user_question,
        lang_code=lang_code,
        system_prompt_base=system_prompt,
    )

    letter = extract_option_letter(raw_answer)
    return letter, raw_answer


def extract_option_letter(text: str) -> str:
    if not text:
        return "A"

    text = text.strip().upper()
    m = re.match(r"^\s*([ABCD])\b", text)
    if m:
        return m.group(1)
    for ch in ["A", "B", "C", "D"]:
        if ch in text:
            return ch
    return "A"


In [39]:
def option_to_onehot(letter: str):
    mapping = {
        "A": [1, 0, 0, 0],
        "B": [0, 1, 0, 0],
        "C": [0, 0, 1, 0],
        "D": [0, 0, 0, 1],
    }

    return mapping.get(letter, [0, 0, 0, 0])


In [ ]:
rows = []
for _, row in df_filtered_track_2.iterrows():
    q_id = row["id"]
    question = row["question"]
    options = {
        "A": row["option_A"],
        "B": row["option_B"],
        "C": row["option_C"],
        "D": row["option_D"],
    }

    letter, raw_answer = ask_mcq_with_rag(
        question=question,
        options=options,
        lang_code="ms-SG",
    )

    A, B, C, D = option_to_onehot(letter)
    rows.append({
        "id": q_id,
        "A": A,
        "B": B,
        "C": C,
        "D": D,
    })

pred_df = pd.DataFrame(rows)
pred_df.to_csv("track_2_mcq_prediction_v1_without_local_db.tsv", sep="\t", index=False)
